<a href="https://colab.research.google.com/github/Michael-AI-Dam/Flyrank-ml-internship/blob/main/Copy_of_w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Michael-AI-Dam/Flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 1. My lane as an ML task (type)
*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Scoring / Ranking.**
My lane is CTR / Search Performance Optimization. This isn't classification —
there's no clean yes/no label for "should this page be fixed." It's not
clustering either, since I'm not grouping similar pages, I'm prioritizing them.
It's a **scoring** problem: assign each page an opportunity score so the content
team can rank pages and work down the list, starting with the highest-value fixes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*


**Proxy target: an "opportunity score."**
There's no direct ground-truth label for "worth optimizing" — I can't observe
whether a title/metadata change *would* increase CTR without actually running
the change. So I build a proxy from observed data: the gap between a page's
actual CTR and the average CTR of other pages at a similar average search
position. A large positive gap (position is fine, but CTR is well below peers
at that position) signals an opportunity.

This is a proxy, not an observed outcome — it's inferred from a defined rule
applied to real numbers, not a confirmed "this fix worked" result.

In [ ]:

import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/Michael-AI-Dam/Flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv")

# Check CTR scale first (fraction 0-1, or already a percentage?)
print(df["ctr"].describe())

count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000
Name: ctr, dtype: float64


In [ ]:
df["ctr_by_position_avg"] = df.groupby("position_tier")["ctr"].transform("mean")
df["opportunity_score"] = (df["impressions_last_30d"] *
                            (df["ctr_by_position_avg"] - df["ctr"]).clip(lower=0))

df[["content_id", "position_tier", "ctr", "ctr_by_position_avg", "opportunity_score"]].sort_values(
    "opportunity_score", ascending=False).head(10)

,content_id,position_tier,ctr,ctr_by_position_avg,opportunity_score
7678,content_8451fc6f034d,top_3,0.03,1.483611,245599.125006
3331,content_4a6607efcb46,top_3,0.01,1.483611,180226.986536
26844,content_8c19996aa890,top_3,0.15,1.483611,119308.797299
18870,content_db5989a78dd3,page_1,0.21,0.652467,105659.245878
21819,content_4c36c775b818,top_3,0.41,1.483611,89885.892956
14090,content_44e481c8f55b,top_3,0.65,1.483611,87077.286937
17812,content_aaef01a50def,page_1,0.25,0.652467,68644.294876
21565,content_9532f197bbc8,top_3,0.87,1.483611,67078.060418
3394,content_36ff89c8214e,page_1,0.05,0.652467,64454.885466
6653,content_5fe46e04994d,page_1,0.14,0.652467,61901.348864


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Success metric
*One metric you can defend. What number means 'good'?*

**Metric: Precision@50.**
Of the top 50 pages the score ranks as highest-opportunity, what fraction
genuinely have CTR meaningfully below their position-tier peers (not just noise)?
This matters more than a generic error metric because the content team will only
act on a short list — what matters is whether the *top* of that list is worth
their time, not how well the score fits every row in the dataset.

In [ ]:
top_50 = df.sort_values("opportunity_score", ascending=False).head(50)

genuine = (top_50["ctr_by_position_avg"] - top_50["ctr"]) > 0.02
precision_at_50 = genuine.mean()

print("Precision@50 (proxy self-check):", round(precision_at_50, 2))

Precision@50 (proxy self-check): 1.0


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 4. The unit of analysis, as a real dataframe
*Load your lane's slice and show it: one row = one what?*

One row = one page (`content_id`), described by its last-30-day search
performance and its computed opportunity score.

In [ ]:
unit_cols = ["content_id", "impressions_last_30d", "clicks_last_30d",
             "ctr", "avg_position", "position_tier", "opportunity_score"]

df[unit_cols].sort_values("opportunity_score", ascending=False).head(10)

,content_id,impressions_last_30d,clicks_last_30d,ctr,avg_position,position_tier,opportunity_score
7678,content_8451fc6f034d,168958,36,0.03,2.3,top_3,245599.125006
3331,content_4a6607efcb46,122303,10,0.01,2.2,top_3,180226.986536
26844,content_8c19996aa890,89463,251,0.15,2.5,top_3,119308.797299
18870,content_db5989a78dd3,238796,501,0.21,5.4,page_1,105659.245878
21819,content_4c36c775b818,83723,548,0.41,2.3,top_3,89885.892956
14090,content_44e481c8f55b,104458,623,0.65,1.4,top_3,87077.286937
17812,content_aaef01a50def,170559,435,0.25,5.4,page_1,68644.294876
21565,content_9532f197bbc8,109317,1176,0.87,2.0,top_3,67078.060418
3394,content_36ff89c8214e,106985,35,0.05,7.3,page_1,64454.885466
6653,content_5fe46e04994d,120791,220,0.14,4.2,page_1,61901.348864


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML beats a fixed rule here
*What makes the pattern too messy for an if-statement?*

A fixed rule (e.g. "impressions > 1,000 and CTR < 5%") ignores context: expected
CTR varies a lot by position. A page at position 8 with 4% CTR may be performing
*above* expectation for that position, while a page at position 2 with the same
4% CTR is badly underperforming. A flat threshold can't tell these apart — it
would flag the wrong page and miss the real opportunity. Comparing each page
against its position-tier peers (as the opportunity score does) captures that
context, which a single if-statement cannot.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.